# MOUSE configuration 166: capillary-background correction map

> **Supplementary poster/testing notebook.** Original processing date: **2026-09-15**; MoDaCor version: **1.8.0**. This is not part of the core MOUSE correction example.

Poster-ready visualization of MoDaCor's attenuation-aware filled/empty-capillary correction for a water-filled borosilicate capillary. The geometry is read from the MOUSE configuration-166 NeXus file; the correction itself uses the same concentric-cylinder numerical kernels as `CapillarySampleContainerCorrection`.

The displayed quantity is the conventional percentage change in the empty-capillary contribution before subtraction,

$$\Delta_{bg}(d)=100\left[s_{bg}(d)-1\right],\qquad s_{bg}(d)=A_{c,sc}(d)/A_{c,c}(d).$$

Thus zero means that the empty-capillary image is subtracted unchanged, while a negative value means that attenuation by the filled capillary reduces the amount subtracted.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("BAM/MOUSE")
DATA_FILE = PROJECT_DIR / "data" / "MOUSE_20260903_2_166_stacked_modacor.nxs"
FIGURE_DIR = PROJECT_DIR / "work" / "supplementary" / "poster_2026" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

import h5py
import matplotlib.pyplot as plt
import numpy as np
import xraylib

from modacor.geometry import ConcentricCylinderGeometry
from modacor.models.attenuation import (
    adaptive_attenuation_factors_on_grid,
    beam_chord_quadrature,
    direct_beam_transmission,
)
from modacor.models.attenuation.beam_profiles import gaussian_beam_profile


## Physical assumptions

The capillary is horizontal (`+x`) for this poster visualization. The Gaussian beam widths are inferred from the direct-beam ellipse stored with configuration 166 and converted using the detector pixel pitch. This is suitable for illustrating the angular structure, but a quantitative correction should use a calibrated sample-plane beam profile.

The unspecified borosilicate grade is represented by a Pyrex-like oxide mixture (mass fractions shown below) at 2.23 g cm⁻³. Replace these values when the capillary composition or density is known.


In [ ]:
INNER_DIAMETER_M = 1.00e-3
WALL_THICKNESS_M = 20.0e-6
CAPILLARY_AXIS = np.array([1.0, 0.0, 0.0])  # horizontal in the detector plane
INCIDENT_DIRECTION = np.array([0.0, 0.0, 1.0])

WATER_DENSITY_G_CM3 = 1.000
BOROSILICATE_DENSITY_G_CM3 = 2.23
BOROSILICATE_OXIDE_MASS_FRACTIONS = {
    "SiO2": 0.806,
    "B2O3": 0.130,
    "Na2O": 0.040,
    "Al2O3": 0.024,
}

RELATIVE_TOLERANCE = 3.0e-4
WALL_CHORD_ORDER = 32


## Read the configuration-166 geometry and attenuation coefficients

The detector coordinates reproduce the `PixelCoordinates3D` centre-of-pixel convention and MOUSE basis vectors in `pipelines/MOUSE_solids.yaml`. Linear attenuation coefficients are evaluated at the photon energy stored in the file (Cu Kα, about 8.048 keV).


In [ ]:
def scalar(h5, path):
    return float(np.asarray(h5[path][()]).reshape(-1)[0])

with h5py.File(DATA_FILE, "r") as h5:
    configuration = int(h5["/entry1/instrument/configuration"][()])
    detector_shape = h5["/entry1/instrument/detector00/data"].shape[-2:]
    detector_z_m = scalar(h5, "/entry1/instrument/detector00/transformations/det_x")
    detector_x0_m = scalar(h5, "/entry1/instrument/detector00/transformations/det_y")
    detector_y0_m = scalar(h5, "/entry1/instrument/detector00/transformations/det_z")
    sample_z_m = 1e-3 * scalar(h5, "/entry1/sample/transformations/sample_x")
    pitch_fast_m = scalar(h5, "/entry1/instrument/detector00/x_pixel_size")
    pitch_slow_m = scalar(h5, "/entry1/instrument/detector00/y_pixel_size")
    energy_kev = 1e-3 * scalar(h5, "/entry1/instrument/detector00/detectorSpecific/photon_energy")
    wavelength_nm = scalar(h5, "/entry1/sample/beam/incident_wavelength")
    sigma_slow_m = pitch_slow_m * scalar(h5, "/entry1/processing/direct_beam_profile/beam_analysis/sigma_minor")
    sigma_fast_m = pitch_fast_m * scalar(h5, "/entry1/processing/direct_beam_profile/beam_analysis/sigma_major")
    beam_rotation_rad = scalar(h5, "/entry1/processing/direct_beam_profile/beam_analysis/theta")

rows = np.arange(detector_shape[0], dtype=float) + 0.5
columns = np.arange(detector_shape[1], dtype=float) + 0.5
coord_x = np.broadcast_to(detector_x0_m - columns[None, :] * pitch_fast_m, detector_shape)
coord_y = np.broadcast_to(detector_y0_m - rows[:, None] * pitch_slow_m, detector_shape)
coord_z = np.full(detector_shape, detector_z_m)
detector_grid = np.stack(np.broadcast_arrays(coord_x, coord_y, coord_z), axis=-1)

sample_position = np.array([0.0, 0.0, sample_z_m])
scattered_rays = detector_grid - sample_position
scattered_directions = scattered_rays / np.linalg.norm(scattered_rays, axis=-1, keepdims=True)
wave_number_nm_inv = 2.0 * np.pi / wavelength_nm
q_vector = wave_number_nm_inv * (scattered_directions - INCIDENT_DIRECTION)
q_x_nm_inv = q_vector[..., 0]
q_y_nm_inv = q_vector[..., 1]

water_mu_m_inv = xraylib.CS_Total_CP("H2O", energy_kev) * WATER_DENSITY_G_CM3 * 100.0
borosilicate_mass_mu_cm2_g = sum(
    fraction * xraylib.CS_Total_CP(oxide, energy_kev)
    for oxide, fraction in BOROSILICATE_OXIDE_MASS_FRACTIONS.items()
)
borosilicate_mu_m_inv = borosilicate_mass_mu_cm2_g * BOROSILICATE_DENSITY_G_CM3 * 100.0

capillary_centre = sample_position
geometry = ConcentricCylinderGeometry(
    radii=np.array([INNER_DIAMETER_M / 2.0, INNER_DIAMETER_M / 2.0 + WALL_THICKNESS_M]),
    axis=CAPILLARY_AXIS,
    centre=capillary_centre,
)
beam_profile = gaussian_beam_profile(
    standard_deviations=[sigma_slow_m, sigma_fast_m],
    quadrature_order=[8, 8],
    truncation_sigma=[4.0, 4.0],
    centre=capillary_centre,
    incident_direction=INCIDENT_DIRECTION,
    rotation=beam_rotation_rad,
)

print(f"MOUSE configuration: {configuration}; detector: {detector_shape[1]} × {detector_shape[0]} pixels")
print(f"Energy: {energy_kev:.4f} keV")
print(f"mu(water): {water_mu_m_inv:.1f} m^-1; mu(borosilicate): {borosilicate_mu_m_inv:.1f} m^-1")
print(f"Beam sigma (slow, fast): {1e6*sigma_slow_m:.1f}, {1e6*sigma_fast_m:.1f} µm")


## Evaluate the coupled filled/empty wall model

The filled- and empty-capillary wall factors are evaluated together so their ratio shares one adaptive detector mesh.


In [ ]:
wall_points, wall_weights = beam_chord_quadrature(
    geometry=geometry, beam_points=beam_profile.points, beam_weights=beam_profile.weights,
    phase_index=1, incident_direction=INCIDENT_DIRECTION, chord_order=WALL_CHORD_ORDER,
)

common = dict(
    geometry=geometry, detector_position_grid=detector_grid, active_mask=None,
    incident_direction=INCIDENT_DIRECTION, relative_tolerance=RELATIVE_TOLERANCE,
    absolute_tolerance=1e-12, max_depth=11, detector_chunk_size=128,
)
wall_result = adaptive_attenuation_factors_on_grid(
    attenuation_coefficients=np.array([
        [water_mu_m_inv, borosilicate_mu_m_inv],
        [0.0, borosilicate_mu_m_inv],
    ]),
    scattering_points=wall_points, volume_weights=wall_weights, **common,
)

wall_filled_attenuation, wall_empty_attenuation = wall_result.factors
background_subtraction_scale = wall_filled_attenuation / wall_empty_attenuation

filled_transmission = direct_beam_transmission(
    geometry=geometry, attenuation_coefficients=[water_mu_m_inv, borosilicate_mu_m_inv],
    beam_points=beam_profile.points, beam_weights=beam_profile.weights,
    incident_direction=INCIDENT_DIRECTION,
)
empty_transmission = direct_beam_transmission(
    geometry=geometry, attenuation_coefficients=[0.0, borosilicate_mu_m_inv],
    beam_points=beam_profile.points, beam_weights=beam_profile.weights,
    incident_direction=INCIDENT_DIRECTION,
)
print(f"Background scale: {background_subtraction_scale.min():.4f}–{background_subtraction_scale.max():.4f}")
print(f"Calculated direct transmission: filled {filled_transmission:.4f}; empty {empty_transmission:.4f}")
print(f"Exact detector nodes: wall {wall_result.evaluated_mask.sum():,}")


## Poster figure

The exact scattered-ray components are used as the curvilinear $q_x,q_y$ coordinates. Sparse detector-row and detector-column traces are overlaid to make the small departure from a rectilinear mapping explicit. The map uses the requested `plasma` colormap; PNG (300 dpi) and scalable SVG versions are saved below `work/supplementary/poster_2026/figures/`.


In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 13,
    "axes.titlesize": 18, "axes.labelsize": 15,
    "xtick.labelsize": 12, "ytick.labelsize": 12,
})
background_change_percent = 100.0 * (background_subtraction_scale - 1.0)

fig, ax = plt.subplots(figsize=(7.8, 6.7), constrained_layout=True)
image = ax.pcolormesh(
    q_x_nm_inv, q_y_nm_inv, background_change_percent, shading="auto",
    cmap="plasma", rasterized=True,
)

# These traces show the exact detector-to-q mapping; at MOUSE angles their curvature is subtle.
mapping_stride = 128
for row in range(0, detector_shape[0], mapping_stride):
    ax.plot(q_x_nm_inv[row, :], q_y_nm_inv[row, :], color="white", lw=0.35, alpha=0.16)
for column in range(0, detector_shape[1], mapping_stride):
    ax.plot(q_x_nm_inv[:, column], q_y_nm_inv[:, column], color="white", lw=0.35, alpha=0.16)

ax.axhline(0, color="white", lw=0.8, alpha=0.38)
ax.axvline(0, color="white", lw=0.8, alpha=0.38)
ax.plot(0, 0, marker="+", ms=11, mew=1.7, color="white")
axis_arrow_end = 0.28 * max(abs(q_x_nm_inv.min()), abs(q_x_nm_inv.max()))
ax.annotate(
    "capillary axis", xy=(axis_arrow_end, 0), xytext=(0, 0),
    ha="left", va="bottom", color="white", fontsize=11,
    arrowprops={"arrowstyle": "-|>", "color": "white", "lw": 1.3},
)
ax.set_xlabel(r"$q_x$ (nm$^{-1}$)")
ax.set_ylabel(r"$q_y$ (nm$^{-1}$)")
ax.set_aspect("equal")
colorbar = fig.colorbar(image, ax=ax, pad=0.025, fraction=0.055)
colorbar.set_label(r"Change in empty-capillary subtraction  $100(s_{bg}-1)$  (%)")

png_path = FIGURE_DIR / "MOUSE_166_water_capillary_background_change_q_plasma.png"
svg_path = FIGURE_DIR / "MOUSE_166_water_capillary_background_change_q_plasma.svg"
fig.savefig(png_path, dpi=300, facecolor="white")
fig.savefig(svg_path, facecolor="white")
plt.show()
print(png_path)
print(svg_path)
